# Actualización diaria — Proyecto HCHNotebook único para poner al día las 5 fuentes del proyecto:1. TRM (Banco de la República)2. CME Chicago (harina de soya + maíz)3. Boletines BMC (HCH y sustitutos)4. RONI (El Niño / La Niña — NOAA)5. Sacrificio bovino (DANE ESAG)**Antes de correr:** ajusta `RUTA_ESAG` en la celda de sacrificio bovino con la ruta real del archivo en tu equipo.

In [1]:
import asyncio, sqlite3, re, io
import nest_asyncio
from pathlib import Path
from datetime import date, datetime
import pandas as pd
import requests
import yfinance as yf
import pytesseract
from pdf2image import convert_from_path

nest_asyncio.apply()

DB_PATH      = "mercado_hch.db"
CARPETA_PDF  = Path("./boletines_bmc")
TRM_FALLBACK = 4200.0
URL_PORTAL   = "https://bpmapps.bolsamercantil.com.co/Boletines/Boletines.aspx"
TESSERACT    = r"C:\Program Files\Tesseract-OCR\tesseract.exe"
POPPLER_PATH = r"C:\poppler\Library\bin"
pytesseract.pytesseract.tesseract_cmd = TESSERACT
CARPETA_PDF.mkdir(parents=True, exist_ok=True)

print("Configuración base lista")

Configuración base lista


## 1. TRM (Banco de la República)

In [2]:
print("Descargando TRM desde datos.gov.co...")
url_trm = "https://www.datos.gov.co/api/views/mcec-87by/rows.csv?accessType=DOWNLOAD"
df_trm = pd.read_csv(url_trm)
df_trm.columns = ['trm_cop_usd', 'unidad', 'fecha_desde', 'fecha_hasta']
df_trm['fecha'] = pd.to_datetime(df_trm['fecha_desde'], format='%d/%m/%Y')
df_trm = df_trm[['fecha', 'trm_cop_usd']].sort_values('fecha')
df_trm = df_trm[df_trm['fecha'] >= '2022-01-01']

fecha_completa = pd.date_range(df_trm['fecha'].min(), df_trm['fecha'].max(), freq='D')
df_trm = df_trm.set_index('fecha').reindex(fecha_completa).ffill().reset_index()
df_trm.columns = ['fecha', 'trm_cop_usd']
df_trm['fecha'] = df_trm['fecha'].dt.strftime('%Y-%m-%d')

with sqlite3.connect(DB_PATH) as con:
    df_trm.to_sql('trm_diaria', con, if_exists='replace', index=False)

print(f"✅ trm_diaria: {len(df_trm)} días, hasta {df_trm['fecha'].max()}")

Descargando TRM desde datos.gov.co...


✅ trm_diaria: 1684 días, hasta 2026-08-14


## 2. CME Chicago (harina de soya + maíz)

In [3]:
print("Descargando CME Harina de Soya y Grano de Maíz...")
soya = yf.download("ZM=F", start="2022-01-01", end=date.today().isoformat(), auto_adjust=True, progress=False)
maiz = yf.download("ZC=F", start="2022-01-01", end=date.today().isoformat(), auto_adjust=True, progress=False)

soya_close = soya[('Close', 'ZM=F')]
maiz_close = maiz[('Close', 'ZC=F')]

df_cme = pd.DataFrame({
    'fecha': soya_close.index.strftime('%Y-%m-%d'),
    'harina_soya_cme_usd_ton': (soya_close.values / 0.907185).round(2),
    'grano_maiz_cme_usd_ton': (maiz_close.values / 100 / 25.4 * 1000).round(2)
})

with sqlite3.connect(DB_PATH) as con:
    df_cme.to_sql('cme_diario', con, if_exists='replace', index=False)

print(f"✅ cme_diario: {len(df_cme)} días, hasta {df_cme['fecha'].max()}")

Descargando CME Harina de Soya y Grano de Maíz...


✅ cme_diario: 1158 días, hasta 2026-08-13


## 3. Boletines BMC (HCH y sustitutos)Definición de funciones + descarga/actualización.

In [4]:
with sqlite3.connect(DB_PATH) as _con:
    _df_trm = pd.read_sql("SELECT fecha, trm_cop_usd FROM trm_diaria", _con)
TRM_POR_FECHA = dict(zip(_df_trm['fecha'], _df_trm['trm_cop_usd']))

def obtener_trm(fecha_str):
    return TRM_POR_FECHA.get(fecha_str, TRM_FALLBACK)

PRODUCTOS_OBJETIVO = [
    "HARINA DE CARNE Y HUESO", "SUBPRODUCTOS CARNICOS", "SEBO DE RES",
    "HARINA DE HUESO", "FOSFATO MONODICALCICO", "HARINA DE PLUMA",
    "HARINA DE VISCERAS", "HARINA DE SANGRE", "TORTA DE SOYA IMPORTADA",
    "ALIMENTO CONCENTRADO PARA AVES", "ALIMENTO CONCENTRADO PARA CERDOS",
    "ALIMENTO CONCENTRADO PARA GATOS", "ALIMENTO CONCENTRADO PARA PERROS"
]

NOMBRE_CORTO = {
    "HARINA DE CARNE Y HUESO": "HCH",
    "SUBPRODUCTOS CARNICOS": "subproductos_carnicos",
    "SEBO DE RES": "sebo_res",
    "HARINA DE HUESO": "harina_hueso",
    "HARINA DE VISCERAS": "harina_visceras",
    "HARINA DE SANGRE": "harina_sangre",
    "HARINA DE PLUMA": "harina_pluma_sangre",
    "TORTA DE SOYA IMPORTADA": "torta_soya_importada",
    "FOSFATO MONODICALCICO": "fosfato_monodicalcico",
    "ALIMENTO CONCENTRADO PARA AVES": "concentrado_aves",
    "ALIMENTO CONCENTRADO PARA CERDOS": "concentrado_cerdos",
    "ALIMENTO CONCENTRADO PARA GATOS": "concentrado_gatos",
    "ALIMENTO CONCENTRADO PARA PERROS": "concentrado_perros"
}

MESES_ES = {
    "ENERO":1,"FEBRERO":2,"MARZO":3,"ABRIL":4,"MAYO":5,"JUNIO":6,
    "JULIO":7,"AGOSTO":8,"SEPTIEMBRE":9,"OCTUBRE":10,"NOVIEMBRE":11,"DICIEMBRE":12
}

def extraer_fecha_y_rueda(nombre_pdf):
    nombre = Path(nombre_pdf).stem.upper()
    m_rueda = re.search(r'N-[_\s]*(\d+)', nombre)
    n_rueda = int(m_rueda.group(1)) if m_rueda else None
    m_fecha = re.search(r'DEL[_\s]+(\d{1,2})[_\s]+DE[_\s]+(\w+)[_\s]+DE[_\s]+(\d{4})', nombre)
    if m_fecha:
        dia, mes, anio = int(m_fecha.group(1)), MESES_ES.get(m_fecha.group(2), 0), int(m_fecha.group(3))
        if mes:
            return f"{anio:04d}-{mes:02d}-{dia:02d}", n_rueda
    return None, n_rueda

def a_float(s):
    s = s.strip()
    if ',' in s and '.' in s:
        s = s.replace('.', '').replace(',', '.') if s.index('.') < s.index(',') else s.replace(',', '')
    elif ',' in s:
        s = s.replace(',', '')
    elif '.' in s:
        partes = s.split('.')
        if len(partes[-1]) == 3:
            s = s.replace('.', '')
    return float(s)

def parsear_linea_bmc(linea, fecha_boletin=None):
    linea = linea.strip()
    if not linea:
        return None
    match_prod = next((o for o in PRODUCTOS_OBJETIVO if o in linea.upper()), None)
    if not match_prod:
        return None
    m_um = re.search(r'\b(KG|LT|UND|LOTE)\b', linea.upper())
    unidad = m_um.group(1) if m_um else ""
    resto = linea[m_um.end():] if m_um else linea
    numeros = re.findall(r'\d{1,3}(?:,\d{3})*(?:\.\d+)?|\d+(?:\.\d+)?', resto)
    if len(numeros) < 3:
        return None
    try:
        cantidad_kg, total_cop, precio_cop = a_float(numeros[0]), a_float(numeros[1]), a_float(numeros[2])
        n_subast = int(float(numeros[3])) if len(numeros) > 3 else None
    except:
        return None
    trm_usada = obtener_trm(fecha_boletin) if fecha_boletin else TRM_FALLBACK
    precio_usd_ton = round((precio_cop / trm_usada) * 1000, 2) if precio_cop else None
    return {
        "producto_bmc": match_prod,
        "producto": NOMBRE_CORTO.get(match_prod, match_prod.lower()),
        "unidad": unidad, "cantidad_kg": cantidad_kg, "total_cop": total_cop,
        "precio_cop_kg": precio_cop, "precio_usd_ton": precio_usd_ton,
        "n_subastadores": n_subast, "trm": trm_usada
    }

def extraer_pdf_completo(ruta_pdf):
    fecha, n_rueda = extraer_fecha_y_rueda(ruta_pdf)
    paginas = convert_from_path(str(ruta_pdf), dpi=200, poppler_path=POPPLER_PATH)
    registros = []
    for pag in paginas:
        for linea in pytesseract.image_to_string(pag, lang='spa').split('\n'):
            r = parsear_linea_bmc(linea, fecha_boletin=fecha)
            if r:
                r.update({'fecha': fecha, 'n_rueda': n_rueda, 'archivo_pdf': Path(ruta_pdf).name})
                registros.append(r)
    df = pd.DataFrame(registros)
    return df.drop_duplicates(subset=['producto_bmc']).to_dict('records') if not df.empty else []

def guardar_en_db(registros):
    if not registros:
        return 0
    nuevos = 0
    with sqlite3.connect(DB_PATH) as con:
        for r in registros:
            try:
                con.execute("""INSERT OR IGNORE INTO bmc_precios
                    (fecha, n_rueda, producto_bmc, producto, unidad, cantidad_kg, total_cop,
                     precio_cop_kg, precio_usd_ton, trm, n_subastadores, archivo_pdf)
                    VALUES (:fecha,:n_rueda,:producto_bmc,:producto,:unidad,:cantidad_kg,:total_cop,
                     :precio_cop_kg,:precio_usd_ton,:trm,:n_subastadores,:archivo_pdf)""", r)
                nuevos += con.execute("SELECT changes()").fetchone()[0]
            except Exception as e:
                print(f"  ⚠️  {e}")
        con.commit()
    return nuevos

async def descargar_boletines(fecha_desde_str, fecha_hasta_str=None):
    from playwright.async_api import async_playwright
    if fecha_hasta_str is None:
        fecha_hasta_str = date.today().strftime("%d/%m/%Y")

    fecha_desde_dt = pd.to_datetime(fecha_desde_str, format="%d/%m/%Y")
    fecha_hasta_dt = pd.to_datetime(fecha_hasta_str, format="%d/%m/%Y")

    ya_en_disco = {p.name for p in CARPETA_PDF.glob("*.pdf")}
    descargados = []

    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        page = await (await browser.new_context(accept_downloads=True)).new_page()
        await page.goto(URL_PORTAL, wait_until="networkidle", timeout=30000)
        await page.wait_for_timeout(2000)
        await page.select_option('#selBoletin', value="1180")
        await page.wait_for_timeout(3000)

        detener = False
        while not detener:
            filas = await page.query_selector_all("table tbody tr")

            for fila in filas:
                nombre_pdf = await fila.get_attribute('nombredocumento')
                if not nombre_pdf:
                    continue

                fecha_str, _ = extraer_fecha_y_rueda(nombre_pdf)
                if fecha_str is None:
                    continue

                fecha_dt = pd.to_datetime(fecha_str)

                if fecha_dt < fecha_desde_dt:
                    detener = True
                    break

                if fecha_dt > fecha_hasta_dt:
                    continue

                if nombre_pdf in ya_en_disco:
                    continue

                btn = await fila.query_selector('#btnDocumento')
                if not btn:
                    continue
                try:
                    async with page.expect_download(timeout=20000) as dl_info:
                        await btn.click()
                    await (await dl_info.value).save_as(str(CARPETA_PDF / nombre_pdf))
                    descargados.append(nombre_pdf)
                    ya_en_disco.add(nombre_pdf)
                    print(f"  ✅ {nombre_pdf[:70]}")
                    await page.wait_for_timeout(800)
                except Exception as e:
                    print(f"  ⚠️  {nombre_pdf[:50]} → {e}")

            if detener:
                break

            siguiente = await page.query_selector('a.paginate_button.next:not(.disabled)')
            if not siguiente:
                break
            await siguiente.click()
            await page.wait_for_timeout(2000)

        await browser.close()

    print(f"🎯 Detenido en fecha límite {fecha_desde_str} — {len(descargados)} PDFs nuevos")
    return descargados

print("✅ Funciones BMC listas (con corte real por fecha)")

✅ Funciones BMC listas (con corte real por fecha)


In [5]:
ultima_fecha_bd = pd.read_sql("SELECT MAX(fecha) as f FROM bmc_precios", sqlite3.connect(DB_PATH))['f'].iloc[0]
print(f"Última fecha BMC en BD: {ultima_fecha_bd}")

nuevos_pdfs = await descargar_boletines(pd.to_datetime(ultima_fecha_bd).strftime("%d/%m/%Y"))

if nuevos_pdfs:
    for nombre in nuevos_pdfs:
        n = guardar_en_db(extraer_pdf_completo(str(CARPETA_PDF / nombre)))
        print(f"✅ {nombre[:60]} → {n} filas")
else:
    print("ℹ️  No hay boletines nuevos")

Última fecha BMC en BD: 2026-08-12


  ✅ BDR_148_BOLETÍN DIARIO DE RUEDA N- 148 DEL 13 DE AGOSTO DE 2026.pdf


🎯 Detenido en fecha límite 12/08/2026 — 1 PDFs nuevos


✅ BDR_148_BOLETÍN DIARIO DE RUEDA N- 148 DEL 13 DE AGOSTO DE 2 → 5 filas


## 4. RONI — El Niño / La Niña (NOAA)

In [6]:
URL_RONI = "https://www.cpc.ncep.noaa.gov/data/indices/RONI.ascii.txt"
SEASON_TO_MONTH = {'DJF':1,'JFM':2,'FMA':3,'MAM':4,'AMJ':5,'MJJ':6,'JJA':7,'JAS':8,'ASO':9,'SON':10,'OND':11,'NDJ':12}

resp = requests.get(URL_RONI, timeout=15)
resp.raise_for_status()
df_roni_raw = pd.read_csv(io.StringIO(resp.text), sep=r"\s+")
df_roni_raw['mes'] = df_roni_raw['SEAS'].map(SEASON_TO_MONTH)
df_roni_raw['anio_mes'] = df_roni_raw['YR'].astype(str) + '-' + df_roni_raw['mes'].astype(str).str.zfill(2)
df_roni = df_roni_raw[['anio_mes', 'ANOM']].rename(columns={'ANOM': 'roni'})

with sqlite3.connect(DB_PATH) as con:
    df_roni.to_sql("roni_mensual", con, if_exists="replace", index=False)

print(f"✅ roni_mensual: {len(df_roni)} meses, hasta {df_roni['anio_mes'].max()}")

✅ roni_mensual: 918 meses, hasta 2026-06


## 5. Sacrificio bovino (DANE ESAG)⚠️

**Ajusta `RUTA_ESAG` con la ruta real del archivo en tu equipo antes de correr esta celda.**

In [7]:
RUTA_ESAG = r"C:\Users\nvind\OneDrive\Documentos\PROYECTO\Portafolio\Proyecto hna_carna_hueso\series-hist-ESAG-Itrim2026.xls"

def parsear_fecha_esag(valor):
    meses_es = {'ene':1,'feb':2,'mar':3,'abr':4,'may':5,'jun':6,'jul':7,'ago':8,'sep':9,'oct':10,'nov':11,'dic':12}
    try:
        return pd.to_datetime(valor)
    except:
        try:
            texto = str(valor).strip().lower().replace('pr','').strip()
            partes = texto.split('-')
            if len(partes) == 2:
                mes, anio_str = meses_es.get(partes[0].strip()), partes[1].strip()
                if mes and anio_str.isdigit() and len(anio_str) == 4:
                    return pd.Timestamp(year=int(anio_str), month=mes, day=1)
            return pd.NaT
        except:
            return pd.NaT

def parsear_numero_esag(valor):
    try:
        if pd.isna(valor):
            return None
        texto = str(valor).strip()
        if any(c.isalpha() for c in texto) or texto in ('', 'nan'):
            return None
        if '.' in texto:
            decimales = len(texto.split('.')[-1])
            return int(texto.replace('.', '')) if decimales == 3 else int(round(float(texto)))
        return int(float(texto))
    except (ValueError, TypeError):
        return None

def cargar_esag_robusto(ruta, sheet, col_nombre):
    df_raw = pd.read_excel(ruta, sheet_name=sheet, header=None, engine='xlrd')
    datos = df_raw.iloc[10:, [0, 1]].copy()
    datos.columns = ['fecha_raw', col_nombre]
    datos = datos[datos['fecha_raw'].notna()].copy()
    datos['fecha'] = datos['fecha_raw'].apply(parsear_fecha_esag)
    datos[col_nombre] = datos[col_nombre].apply(parsear_numero_esag)
    return datos[datos['fecha'].notna() & datos[col_nombre].notna()][['fecha', col_nombre]].reset_index(drop=True)

vacunos = cargar_esag_robusto(RUTA_ESAG, 'Cuadro 1', 'vacunos_cabezas')
bufalinos = cargar_esag_robusto(RUTA_ESAG, 'Cuadro 2', 'bufalinos_cabezas')
porcinos = cargar_esag_robusto(RUTA_ESAG, 'Cuadro 3', 'porcinos_cabezas')  # <- NUEVO: antes faltaba

df_esag = pd.merge(vacunos, bufalinos, on='fecha', how='outer')
df_esag = pd.merge(df_esag, porcinos, on='fecha', how='outer')  # <- NUEVO
df_esag = df_esag.sort_values('fecha').reset_index(drop=True)
df_esag['potencial_hch'] = df_esag['vacunos_cabezas'].fillna(0) + df_esag['bufalinos_cabezas'].fillna(0)
df_esag['fuente'] = 'DANE_ESAG'

with sqlite3.connect(DB_PATH) as con:
    df_esag.to_sql("sacrificio_bovino", con, if_exists="replace", index=False)

print(f"Actualizado sacrificio_bovino: {len(df_esag)} meses, hasta {df_esag['fecha'].max().date()}")
print(f"   Con porcinos_cabezas: {df_esag['porcinos_cabezas'].notna().sum()} meses")


Actualizado sacrificio_bovino: 210 meses, hasta 2026-03-01
   Con porcinos_cabezas: 210 meses


## ✅ Resumen final

In [8]:
print("="*60)
print("ACTUALIZACIÓN COMPLETA — resumen")
print("="*60)
con = sqlite3.connect(DB_PATH)
for tabla, col_fecha in [
    ("trm_diaria", "fecha"), ("cme_diario", "fecha"), ("bmc_precios", "fecha"),
    ("roni_mensual", "anio_mes"), ("sacrificio_bovino", "fecha")
]:
    r = pd.read_sql(f"SELECT COUNT(*) AS n, MAX({col_fecha}) AS ultima FROM {tabla}", con)
    print(f"{tabla:20s} → {r['n'].iloc[0]:>6,} filas | última: {r['ultima'].iloc[0]}")
con.close()

ACTUALIZACIÓN COMPLETA — resumen
trm_diaria           →  1,684 filas | última: 2026-08-14
cme_diario           →  1,158 filas | última: 2026-08-13
bmc_precios          →  9,352 filas | última: 2026-08-13
roni_mensual         →    918 filas | última: 2026-06
sacrificio_bovino    →    210 filas | última: 2026-03-01 00:00:00
